# Preparation of a MIMt V3–V4 taxonomic classifier

This notebook prepares a **MIMt Naive Bayes taxonomic classifier** for the V3–V4 region of the 16S rRNA gene using **MIMt-16S** and the primer pair **341F/806R**.

The workflow follows the same general sequence as the original notebook:

1. configure paths and execution parameters;
1. obtain and import MIMt-16S reference sequences;
1. obtain and import MIMt-16S taxonomy mapping;
1. generate fixed-rank taxonomy with RESCRIPt;
1. extract the V3–V4 region in silico;
1. train Naive Bayes classifiers;
1. classify the reference sequences;
1. compare expected and observed taxonomy.

The notebook keeps the QIIME 2 commands explicit through `!qiime`, making each step easy to inspect and reproduce.

> **Methodological note:** the final comparison classifies the same reference sequences used to train the classifier. Therefore, it should be interpreted as an internal consistency check rather than as an independent estimate of classifier performance.

### Main references

- Cabezas MP., Fonseca NA., Muñoz-Merida A. *MIMt – A curated 16S rRNA reference database with less redundancy and higher accuracy at species-level identification.* Environmental Microbiome 19, 88 (2024)
- Bokulich NA, Kaehler BD, Rideout JR, et al. *Optimizing taxonomic classification of marker-gene amplicon sequences with QIIME 2's q2-feature-classifier plugin*. Microbiome. 2018.
- Robeson MS II, O'Rourke DR, Kaehler BD, et al. *RESCRIPt: Reproducible sequence taxonomy reference database management*. PLoS Computational Biology. 2021.


## 1. Configuration

The configuration is divided into two categories:

- **Project parameters:** values that define the classifier itself and should remain the same across computational environments.
- **Environment parameters:** paths and resource settings that depend on the machine where the notebook is executed.

This separation avoids embedding workstation- or server-specific paths in the scientific workflow.


### 1.1 Set project and environment parameters

In [1]:
from pathlib import Path
import os

# ============================================================
# PROJECT PARAMETERS
# ============================================================

MIMT_MARKER = "16S" # [16S, 18S, 23S, 28S, ITS]
MIMT_SOURCE = "M2c" # [M2c, ""]
MIMT_VERSION = "26_03" # YY_MM

FORWARD_PRIMER = "CCTACGGGRSGCAGCAG"
REVERSE_PRIMER = "GGACTACHVGGGTWTCTAAT"

MIN_LENGTH = 350
MAX_LENGTH = 550

# Parallelization
N_JOBS_EXTRACT = 8
N_JOBS_CLASSIFY = 2

# Optional conservative batch size for classify-sklearn
READS_PER_BATCH = 1000


# ============================================================
# ENVIRONMENT PARAMETERS
# ============================================================

# Directory where project files and classifier artifacts are stored.
# Change this path according to the execution environment.
# PROJECT_DIR = Path("/path/to/silva-138.2-dev").resolve()
PROJECT_DIR = Path("/mnt/data/qiime2-classifiers/mimt-dev-M2c").resolve()


# Directory with ample disk space for temporary files.
# This should point to a filesystem with sufficient free space.
# TEMP_DIR = Path("/path/to/large/tmp").resolve()
TEMP_DIR = Path("/mnt/data/tmp").resolve()

# Separate Joblib temporary directory used by classify-sklearn.
JOBLIB_TEMP_DIR = TEMP_DIR / "joblib"

# Optional QIIME 2 cache directory.
QIIME_CACHE = Path(os.getenv("QIIME_CACHE", PROJECT_DIR / "cache" / "qiime"))

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)
JOBLIB_TEMP_DIR.mkdir(parents=True, exist_ok=True)
QIIME_CACHE.mkdir(parents=True, exist_ok=True)

os.environ["TMPDIR"] = str(TEMP_DIR)
os.environ["TMP"] = str(TEMP_DIR)
os.environ["TEMP"] = str(TEMP_DIR)
os.environ["JOBLIB_TEMP_FOLDER"] = str(JOBLIB_TEMP_DIR)

os.chdir(PROJECT_DIR)

print("PROJECT_DIR        :", PROJECT_DIR)
print("TEMP_DIR           :", TEMP_DIR)
print("JOBLIB_TEMP_FOLDER :", JOBLIB_TEMP_DIR)
print("QIIME_CACHE        :", QIIME_CACHE)


PROJECT_DIR        : /mnt/data/qiime2-classifiers/mimt-dev-M2c
TEMP_DIR           : /mnt/data/tmp
JOBLIB_TEMP_FOLDER : /mnt/data/tmp/joblib
QIIME_CACHE        : /mnt/data/qiime2-classifiers/mimt-dev-M2c/cache/qiime


In [2]:
MIMT_BASE_URL = f"https://people.biopolis.pt/bu/mimt/downloads/{MIMT_MARKER}_files"
MINT_FILE_BASENAME = f"MIMt-{MIMT_MARKER}_{MIMT_SOURCE+'_' if MIMT_SOURCE != '' else ''}{MIMT_VERSION}"

# FASTA FILE
FASTA_GZ = f"{MINT_FILE_BASENAME}_tax.fna.gz"
FASTA_FILE = f"{MINT_FILE_BASENAME}_tax.fna"
MIMT_FASTA_URL  = f"{MIMT_BASE_URL}/{FASTA_GZ}"
MIMT_FASTA_FILE = TEMP_DIR / FASTA_FILE

# SEQS QZA FILE
MIMT_SEQS_QZA = PROJECT_DIR / f"{MINT_FILE_BASENAME}_seqs.qza"

# MAP FILE
TAXMAP_GZ = f"{MINT_FILE_BASENAME}.tax.gz"
TAXMAP_FILE = f"{MINT_FILE_BASENAME}.tsv"
MIMT_MAP_URL = f"{MIMT_BASE_URL}/{TAXMAP_GZ}"
MIMT_MAP_FILE = TEMP_DIR / TAXMAP_FILE

# REF TAX QZA
MIMT_REFTAX_QZA = PROJECT_DIR / f"{MINT_FILE_BASENAME}_ref_taxonomy.qza"

# V3V4 Extraction QZA - Stats and Sequences
STATS_QZA = PROJECT_DIR / f"{MINT_FILE_BASENAME}-stats.qza"
SEQS_V3V4_QZA = PROJECT_DIR / f"{MINT_FILE_BASENAME}-v3v4-341f-806r-seqs.qza"

# Classifier QZA
MINT_CLASSIFIER_QZA = PROJECT_DIR / f"{MINT_FILE_BASENAME}-v3v4-341f-806r-nb-classifier.qza"

# Classification QZA
OBSERVED_TAX_QZA = PROJECT_DIR / f"{MINT_FILE_BASENAME}-v3v4-341f-806r-nb-observed-taxonomy.qza"
EVAL_QZV = PROJECT_DIR / f"{MINT_FILE_BASENAME}-v3v4-341f-806r-nb-evaluation.qzv"

### 1.2 Check the execution environment

Large QIIME 2 classifiers may require substantial temporary storage during training, loading, and parallel classification. In particular:

- QIIME 2 may extract artifacts into the temporary directory;
- Joblib may create temporary memory-mapped arrays when `classify-sklearn` uses multiple workers;
- the cache may also consume additional storage.

Before starting the pipeline, confirm that the configured temporary filesystem has sufficient free space.


In [3]:
!echo "TMPDIR=$TMPDIR"
!echo "JOBLIB_TEMP_FOLDER=$JOBLIB_TEMP_FOLDER"
!python -c "import tempfile; print('Python temp directory:', tempfile.gettempdir())"

!df -h "$PROJECT_DIR" "$TEMP_DIR"

TMPDIR=/mnt/data/tmp
JOBLIB_TEMP_FOLDER=/mnt/data/tmp/joblib
Python temp directory: /mnt/data/tmp
Filesystem                   Size  Used Avail Use% Mounted on
/dev/mapper/ubuntu--vg-data  6.9T  282G  6.3T   5% /mnt/data
/dev/mapper/ubuntu--vg-data  6.9T  282G  6.3T   5% /mnt/data


In [4]:
!qiime --version
!python -c "import sklearn, joblib; print('scikit-learn:', sklearn.__version__); print('joblib:', joblib.__version__)"

q2cli version 2026.7.0
Run `qiime info` for more version details.
scikit-learn: 1.7.1
joblib: 1.5.3


## 2. MIMt reference sequences

MIMt provides a curated, non-redundant 16S rRNA reference database for identifying archaeal and bacterial taxa. Its sequences are obtained from well-characterized genomes deposited in NCBI and include complete taxonomic assignments, making the database particularly suitable for accurate species-level taxonomic classification workflows.

The 16S sequence export used here is:

`MIMt-<MARKER>_<SOURCE>_<YY>_<MM>_tax.fna`

If downloading the database directly within the execution environment is unreliable, the reference FASTA and its corresponding taxonomy file can be downloaded manually from the [MIMt website](https://chatgpt.com/c/6a972615-5be4-83e9-9fe3-564a33c3a26a#:~:text=MIMt%20provides%20a,into%20QIIME%202.) and imported into QIIME 2.


In [5]:
print("Download FASTA file from:", MIMT_FASTA_URL)

Download FASTA file from: https://people.biopolis.pt/bu/mimt/downloads/16S_files/MIMt-16S_M2c_26_03_tax.fna.gz


In [6]:
MIMT_FASTA_GZ = TEMP_DIR / FASTA_GZ
!wget --continue \
      --tries=20 \
      --timeout=120 \
      --read-timeout=120 \
      --retry-connrefused \
      --waitretry=10 \
      --output-document={MIMT_FASTA_GZ} \
      {MIMT_FASTA_URL}

--2026-09-01 19:28:57--  https://people.biopolis.pt/bu/mimt/downloads/16S_files/MIMt-16S_M2c_26_03_tax.fna.gz
Resolving people.biopolis.pt (people.biopolis.pt)... 193.137.39.30
Connecting to people.biopolis.pt (people.biopolis.pt)|193.137.39.30|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11193682 (11M) [application/x-gzip]
Saving to: ‘/mnt/data/tmp/MIMt-16S_M2c_26_03_tax.fna.gz’

/mnt/data/tmp/MIMt- 100%[===================>]  10.67M  6.62MB/s    in 1.6s    

2026-09-01 19:29:00 (6.62 MB/s) - ‘/mnt/data/tmp/MIMt-16S_M2c_26_03_tax.fna.gz’ saved [11193682/11193682]



In [7]:
!gunzip -c "$MIMT_FASTA_GZ" > "$MIMT_FASTA_FILE"
!ls -lh "$TEMP_DIR"

total 98M
drwxrwsr-x 2 lauro adm 4.0K Sep  1 19:28 joblib
-rw-rw-r-- 1 lauro adm  87M Sep  1 19:29 MIMt-16S_M2c_26_03_tax.fna
-rw-rw-r-- 1 lauro adm  11M Feb 27  2026 MIMt-16S_M2c_26_03_tax.fna.gz


### 2.3 Import RNA reference sequences


In [8]:
!qiime tools import \
    --type 'FeatureData[Sequence]' \
    --input-path "$MIMT_FASTA_FILE" \
    --output-path "$MIMT_SEQS_QZA"

Imported /mnt/data/tmp/MIMt-16S_M2c_26_03_tax.fna as DNASequencesDirectoryFormat to /mnt/data/qiime2-classifiers/mimt-dev-M2c/MIMt-16S_M2c_26_03_seqs.qza


In [9]:
!qiime tools peek "$MIMT_SEQS_QZA"

!qiime tools validate "$MIMT_SEQS_QZA"

UUID:        79eff8a0-168f-4960-a5c2-beaa6be27bb7
Type:        FeatureData[Sequence]
Data format: DNASequencesDirectoryFormat
Result /mnt/data/qiime2-classifiers/mimt-dev-M2c/MIMt-16S_M2c_26_03_seqs.qza appears to be valid at level=max.


## 3. MIMT taxonomy resources

These files should be placed in `PROJECT_DIR`.


In [10]:
print(MIMT_MAP_URL)

https://people.biopolis.pt/bu/mimt/downloads/16S_files/MIMt-16S_M2c_26_03.tax.gz


In [11]:
MIMT_MAP_GZ = TEMP_DIR / TAXMAP_GZ
!wget --continue \
      --tries=20 \
      --timeout=120 \
      --read-timeout=120 \
      --retry-connrefused \
      --waitretry=10 \
      --output-document={MIMT_MAP_GZ} \
      {MIMT_MAP_URL}

--2026-09-01 19:29:42--  https://people.biopolis.pt/bu/mimt/downloads/16S_files/MIMt-16S_M2c_26_03.tax.gz
Resolving people.biopolis.pt (people.biopolis.pt)... 193.137.39.30
Connecting to people.biopolis.pt (people.biopolis.pt)|193.137.39.30|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1265856 (1.2M) [application/x-gzip]
Saving to: ‘/mnt/data/tmp/MIMt-16S_M2c_26_03.tax.gz’

/mnt/data/tmp/MIMt- 100%[===================>]   1.21M  1.87MB/s    in 0.6s    

2026-09-01 19:29:43 (1.87 MB/s) - ‘/mnt/data/tmp/MIMt-16S_M2c_26_03.tax.gz’ saved [1265856/1265856]



### 3.1 Decompress the taxonomy files

The `-k` option preserves the original compressed files.


In [12]:
!gunzip -c "$MIMT_MAP_GZ" > "$MIMT_MAP_FILE"
!ls -lh "$TEMP_DIR"

total 107M
drwxrwsr-x 2 lauro adm 4.0K Sep  1 19:28 joblib
-rw-rw-r-- 1 lauro adm  87M Sep  1 19:29 MIMt-16S_M2c_26_03_tax.fna
-rw-rw-r-- 1 lauro adm  11M Feb 27  2026 MIMt-16S_M2c_26_03_tax.fna.gz
-rw-rw-r-- 1 lauro adm 1.3M Feb 27  2026 MIMt-16S_M2c_26_03.tax.gz
-rw-rw-r-- 1 lauro adm 8.3M Sep  1 19:29 MIMt-16S_M2c_26_03.tsv
drwxrwxrwt 3 lauro adm 4.0K Sep  1 19:29 qiime2


### 3.2 Import SILVA taxonomy components into QIIME 2


In [13]:
!qiime tools import \
    --type 'FeatureData[Taxonomy]' \
    --input-format HeaderlessTSVTaxonomyFormat \
    --input-path {MIMT_MAP_FILE} \
    --output-path {MIMT_REFTAX_QZA}

Imported /mnt/data/tmp/MIMt-16S_M2c_26_03.tsv as HeaderlessTSVTaxonomyFormat to /mnt/data/qiime2-classifiers/mimt-dev-M2c/MIMt-16S_M2c_26_03_ref_taxonomy.qza


In [14]:
!qiime tools peek "$MIMT_REFTAX_QZA"

!qiime tools validate "$MIMT_REFTAX_QZA"

UUID:        5fc5e520-677d-4504-a7fe-7a6aa9ab2e59
Type:        FeatureData[Taxonomy]
Data format: TSVTaxonomyDirectoryFormat
Result /mnt/data/qiime2-classifiers/mimt-dev-M2c/MIMt-16S_M2c_26_03_ref_taxonomy.qza appears to be valid at level=max.


## 5. Extract the V3–V4 region

The experimental amplicon targets the V3–V4 region of the 16S rRNA gene using primers 341F and 806R.

Primer sequences used in this project:

- **341F:** `CCTACGGGRSGCAGCAG`
- **806R:** `GGACTACHVGGGTWTCTAAT`

The reference sequences are therefore trimmed in silico so that the classifier is trained on the same marker region expected in the sequencing data.

The extraction accepts products between 350 and 550 bp to accommodate expected biological variation around the V3–V4 amplicon.


In [15]:
!qiime feature-classifier extract-reads \
    --i-sequences {MIMT_SEQS_QZA} \
    --p-f-primer {FORWARD_PRIMER} \
    --p-r-primer {REVERSE_PRIMER} \
    --p-read-orientation forward \
    --p-min-length {MIN_LENGTH} \
    --p-max-length {MAX_LENGTH} \
    --p-n-jobs {N_JOBS_EXTRACT} \
    --o-read-extraction-stats {STATS_QZA} \
    --o-reads {SEQS_V3V4_QZA} \
    # --use-cache {QIIME_CACHE} \
    --verbose

Saved FeatureData[Sequence] to: /mnt/data/qiime2-classifiers/mimt-dev-M2c/MIMt-16S_M2c_26_03-v3v4-341f-806r-seqs.qza
Saved ImmutableMetadata to: /mnt/data/qiime2-classifiers/mimt-dev-M2c/MIMt-16S_M2c_26_03-stats.qza


In [16]:
!qiime tools peek {SEQS_V3V4_QZA}

!qiime tools validate {SEQS_V3V4_QZA}


UUID:        40d0fc06-5683-4bdc-94d2-806be1ee95aa
Type:        FeatureData[Sequence]
Data format: DNASequencesDirectoryFormat
Result /mnt/data/qiime2-classifiers/mimt-dev-M2c/MIMt-16S_M2c_26_03-v3v4-341f-806r-seqs.qza appears to be valid at level=max.


## 5. Train Naive Bayes classifiers

The QIIME 2 `fit-classifier-naive-bayes` action trains a scikit-learn-based taxonomic classifier using the extracted V3–V4 reference reads and the corresponding SILVA taxonomy.

The classifier is serialized together with the scikit-learn model, so it is advisable to use it with the same compatible QIIME 2/scikit-learn environment in which it was created.


In [17]:
!qiime feature-classifier fit-classifier-naive-bayes \
    --i-reference-reads {SEQS_V3V4_QZA} \
    --i-reference-taxonomy {MIMT_REFTAX_QZA} \
    --o-classifier {MINT_CLASSIFIER_QZA} \
    # --use-cache {QIIME_CACHE} \
    --verbose


Saved TaxonomicClassifier to: /mnt/data/qiime2-classifiers/mimt-dev-M2c/MIMt-16S_M2c_26_03-v3v4-341f-806r-nb-classifier.qza


In [18]:
!qiime tools peek {MINT_CLASSIFIER_QZA}

!qiime tools validate {MINT_CLASSIFIER_QZA}

UUID:        97666fff-44c9-463f-9e91-2941ef3d230f
Type:        TaxonomicClassifier
Data format: TaxonomicClassiferTemporaryPickleDirFmt
Result /mnt/data/qiime2-classifiers/mimt-dev-M2c/MIMt-16S_M2c_26_03-v3v4-341f-806r-nb-classifier.qza appears to be valid at level=max.


## 8. Classify the reference reads

This step applies the trained classifier back to the extracted SILVA V3–V4 reads.

The purpose is to compare the expected SILVA taxonomy with the taxonomy predicted by the classifier. Because the same sequences were used during training, this is an **internal consistency evaluation**, not an independent validation.

### Temporary storage

`classify-sklearn` uses Joblib for multiprocessing. With multiple workers, Joblib may create large temporary memory-mapped files. `JOBLIB_TEMP_FOLDER` was therefore configured in Section 1 to point to the large temporary filesystem.


### 8.1 Classification


In [19]:
!qiime feature-classifier classify-sklearn \
    --i-classifier {MINT_CLASSIFIER_QZA} \
    --i-reads {SEQS_V3V4_QZA} \
    --p-n-jobs {N_JOBS_CLASSIFY} \
    --p-reads-per-batch {READS_PER_BATCH} \
    --o-classification {OBSERVED_TAX_QZA} \
    # --use-cache {QIIME_CACHE} \
    --verbose


Saved FeatureData[Taxonomy] to: /mnt/data/qiime2-classifiers/mimt-dev-M2c/MIMt-16S_M2c_26_03-v3v4-341f-806r-nb-observed-taxonomy.qza


In [20]:
!qiime tools peek {OBSERVED_TAX_QZA}

!qiime tools validate {OBSERVED_TAX_QZA}


UUID:        8a15dc1d-c9dc-4762-b9d1-162a95be52eb
Type:        FeatureData[Taxonomy]
Data format: TSVTaxonomyDirectoryFormat
Result /mnt/data/qiime2-classifiers/mimt-dev-M2c/MIMt-16S_M2c_26_03-v3v4-341f-806r-nb-observed-taxonomy.qza appears to be valid at level=max.


## 9. Evaluate expected versus observed taxonomy

RESCRIPt `evaluate-classifications` compares the expected reference taxonomy with the taxonomy predicted by the classifier.

The generated `.qzv` files summarize agreement across taxonomic ranks. Because this evaluation uses the training reference sequences, the results should be reported as **training-set/internal consistency metrics**.

For an unbiased estimate of generalization performance, an independent test set or an appropriate cross-validation/hold-out strategy would be required.


### 9.1 Evaluation


In [21]:
!qiime rescript evaluate-classifications \
    --i-expected-taxonomies "$MIMT_REFTAX_QZA" \
    --i-observed-taxonomies "$OBSERVED_TAX_QZA" \
    --p-labels f"{MINT_FILE_BASENAME} V3-V4 341F-806R" \
    --o-evaluation "$EVAL_QZV" \
    # --use-cache "$QIIME_CACHE" \
    --verbose


Saved Visualization to: /mnt/data/qiime2-classifiers/mimt-dev-M2c/MIMt-16S_M2c_26_03-v3v4-341f-806r-nb-evaluation.qzv


In [22]:
!qiime tools peek {EVAL_QZV}

!qiime tools validate {EVAL_QZV}


UUID:        13197731-c53c-4dbe-9fa4-d0fa5533e28f
Type:        Visualization
Result /mnt/data/qiime2-classifiers/mimt-dev-M2c/MIMt-16S_M2c_26_03-v3v4-341f-806r-nb-evaluation.qzv appears to be valid at level=max.


Clean up Temporary Files

In [23]:
!rm -R {TEMP_DIR}

## 10. Inspect final artifacts

On a headless server, `.qzv` files are usually downloaded and opened with **QIIME 2 View**:

https://view.qiime2.org/

The commands below only verify that the final artifacts exist and can be interpreted by QIIME 2.


## 11. Notes for reproducibility

When distributing or reporting this classifier, record at least:

- SILVA release: **138.2**;
- reference collection: **SSU Ref NR99**;
- target region: **16S V3–V4**;
- primers: **341F / 806R**;
- amplicon length filter: **350–550 bp**;
- QIIME 2 version;
- RESCRIPt version;
- q2-feature-classifier version;
- scikit-learn version;
- whether taxonomy was limited to genus or included species labels.

The `.qza` and `.qzv` artifacts preserve QIIME 2 provenance internally, but the notebook provides a human-readable record of the analytical decisions.


## 12. References

**Bokulich NA, Kaehler BD, Rideout JR, et al.** Optimizing taxonomic classification of marker-gene amplicon sequences with QIIME 2's q2-feature-classifier plugin. *Microbiome*. 2018;6:90.  
https://doi.org/10.1186/s40168-018-0470-z

**Robeson MS II, O'Rourke DR, Kaehler BD, et al.** RESCRIPt: Reproducible sequence taxonomy reference database management. *PLoS Computational Biology*. 2021;17:e1009581.  
https://doi.org/10.1371/journal.pcbi.1009581

**Cabezas MP., Fonseca NA., Muñoz-Merida A.** MIMt – A curated 16S rRNA reference database with less redundancy and higher accuracy at species-level identification. *Environmental Microbiome*. 2024;12,88.  
https://doi.org/10.1186/s40793-024-00634-w
